## Imports

In [ ]:
from mmengine.utils import get_git_hash
from mmengine.utils.dl_utils import collect_env as collect_base_env
from mmdet.registry import VISUALIZERS
from pycocotools.coco import COCO

import mmdet
import mmcv

import matplotlib.pyplot as plt

## System Information

In [ ]:
def collect_env():
    """Collect the information of the running environments."""
    env_info = collect_base_env()
    env_info['MMDetection'] = f'{mmdet.__version__}+{get_git_hash()[:7]}'
    return env_info


if __name__ == '__main__':
    for name, val in collect_env().items():
        print(f'{name}: {val}')

## Train

In [ ]:
img = mmcv.imread('datasets/coco/train2017/000000000127.jpg')
plt.figure(figsize=(15, 10))
plt.imshow(mmcv.bgr2rgb(img))
plt.show()

In [ ]:
# Path to load the COCO annotation file
annotation_file = 'datasets/coco/annotations/instances_train2017.json'

# Initialise the COCO object
coco = COCO(annotation_file)

# Get all category tags and corresponding category IDs
categories = coco.loadCats(coco.getCatIds())
category_id_to_name = {cat['id']: cat['name'] for cat in categories}

# Print all category IDs and corresponding category names
for category_id, category_name in category_id_to_name.items():
    print(f"Category ID: {category_id}, Category Name: {category_name}")

In [ ]:
from mmengine import Config
cfg = Config.fromfile('./configs/mask_rcnn/mask-rcnn_r50-caffe_fpn_ms-poly-1x_coco.py')

In [ ]:
from mmengine.runner import set_random_seed

# Modify dataset type and path
cfg.data_root = './datasets/coco/'

cfg.train_dataloader.dataset.ann_file = 'annotations/instances_train2017.json'
cfg.train_dataloader.dataset.data_root = cfg.data_root
cfg.train_dataloader.dataset.data_prefix.img = 'train2017/'
cfg.train_dataloader.dataset.metainfo = None  # COCO classes are loaded automatically

cfg.val_dataloader.dataset.ann_file = 'annotations/instances_val2017.json'
cfg.val_dataloader.dataset.data_root = cfg.data_root
cfg.val_dataloader.dataset.data_prefix.img = 'val2017/'
cfg.val_dataloader.dataset.metainfo = None

cfg.test_dataloader = cfg.val_dataloader

# Modify metric config
cfg.val_evaluator.ann_file = cfg.data_root + 'annotations/instances_val2017.json'
cfg.test_evaluator = cfg.val_evaluator

# Modify num classes of the model in box head and mask head
cfg.model.roi_head.bbox_head.num_classes = 80
cfg.model.roi_head.mask_head.num_classes = 80

# Use pre-trained Mask R-CNN model trained on COCO
# cfg.load_from = 'checkpoints/mask_rcnn_r50_caffe_fpn_mstrain-poly_3x_coco_bbox_mAP-0.408__segm_mAP-0.37_20200504_163245-42aa3d00.pth'

# Set up working dir to save files and logs
cfg.work_dir = './coco_exps_from_scratch'

# Evaluation and checkpoint intervals
cfg.train_cfg.val_interval = 1
cfg.default_hooks.checkpoint.interval = 1

# Adjust learning rate for single GPU
cfg.optim_wrapper.optimizer.lr = 0.02 / 8
cfg.default_hooks.logger.interval = 10

# Train for only 1 epoch
cfg.train_cfg.max_epochs = 1

# Set seed for reproducibility
set_random_seed(0, deterministic=False)

# Add Tensorboard logger
cfg.visualizer.vis_backends.append({"type": 'TensorboardVisBackend'})

# Save config
config = './configs/mask_rcnn/config.py'
with open(config, 'w') as f:
    f.write(cfg.pretty_text)

In [ ]:
# !python tools/train.py {config}

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir coco_exps/

In [ ]:
import mmcv
from mmdet.apis import init_detector, inference_detector
img = mmcv.imread('./datasets/coco/val2017/000000000285.jpg',channel_order='rgb')
checkpoint_file = 'coco_exps_from_scratch/epoch_1.pth'
model = init_detector(cfg, checkpoint_file, device='cpu')
new_result = inference_detector(model, img)
print(new_result)

In [ ]:
import copy

# Make a copy of the visualizer config
vis_cfg = copy.deepcopy(model.cfg.visualizer)

for backend in vis_cfg.vis_backends:
    if isinstance(backend, dict):
        backend.setdefault('save_dir', './outputs')

# Build the visualizer
visualizer = VISUALIZERS.build(vis_cfg)

# Assign dataset metadata
visualizer.dataset_meta = model.dataset_meta

In [ ]:
# show the results
visualizer.add_datasample(
    'new_result',
    img,
    data_sample=new_result,
    draw_gt=False,
    wait_time=0,
    out_file=None,
    pred_score_thr=0.5
)
visualizer.show()